In [0]:
import subprocess
subprocess.check_call(['pip', 'install', 'faker', 'holidays', 'pyyaml'])

#from utils import save_to_parquet
import numpy as np, pandas as pd, random
from faker import Faker
import holidays

np.random.seed(42)
random.seed(42)
fake = Faker()
Faker.seed(42)

# Get catalog and schema from notebook widgets / job parameters
# Widgets are automatically created from base_parameters in the workflow
# For local or ad‑hoc runs, defaults are used
dbutils.widgets.text("CATALOG_NAME", "your_workspace_catalog", "Catalog")
dbutils.widgets.text("SCHEMA_NAME", "glucosphere", "Schema")

try:
    CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
    SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
except Exception as e:
    raise RuntimeError(f"Widget lookup failed, catalog/schema not passed to notebook: {e}")

print(f"Using catalog: {CATALOG_NAME}, schema: {SCHEMA_NAME}")

# Ensure Unity Catalog structure exists for data storage
print("Setting up Unity Catalog structure...")
try:
    # Ensure catalog exists (if you don't have permission to create catalogs,
    # comment this out and create the catalog once via SQL instead)
    #spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG_NAME}")
    print(f"✓ Catalog '{CATALOG_NAME}' ready")
    
    # Ensure schema exists
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}")
    print(f"✓ Schema '{SCHEMA_NAME}' ready")
    
    # NOW set current catalog and schema (after creating them)
    spark.sql(f"USE CATALOG {CATALOG_NAME}")
    spark.sql(f"USE SCHEMA {SCHEMA_NAME}")
    
    # Ensure volume exists
    spark.sql(f"""
        CREATE VOLUME IF NOT EXISTS {CATALOG_NAME}.{SCHEMA_NAME}.pipeline_data
        COMMENT 'Landing zone for MedTech raw data files'
    """)
    print("✓ Volume 'pipeline_data' ready")
    
    # Create subdirectories for all datasets using dbutils
    volume_base = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/pipeline_data"
    datasets = [
        "raw_patient_registry",
        "raw_device_telemetry_stream"
    ]
    
    for dataset in datasets:
        dataset_path = f"{volume_base}/{dataset}"
        try:
            dbutils.fs.mkdirs(dataset_path)
            print(f"✓ Volume directory '{dataset}' created")
        except Exception as dir_error:
            print(f"⚠ Could not create directory {dataset}: {dir_error}")
    
except Exception as e:
    print(f"⚠ Could not create Unity Catalog structure: {e}")
    print("  You may need to create catalog/schema/volume manually before running this script")


# ─── Config: copy of 04's Config class (yaml + widget override + auto-resolve) ───
# Same pattern as Data_DataGen_ModelForecast/04_pseudo_data_forecast_modeling.py:73-170.
# Needed here so firmware-event timestamps can derive from cfg.demo_week_start
# (instead of hardcoded Jan 2026 literals that drift past the sliding 7-day window).
dbutils.widgets.text("DEMO_WEEK_START", "", "Demo Week Start override (empty = use YAML 'auto'/specific date)")
dbutils.widgets.text("CONFIG_FILE", "../../configs/baseline_config.yaml", "Config File (relative to notebook dir)")

import yaml
from datetime import datetime, timedelta

class Config:
    def __init__(self, yaml_path, env, widget_overrides=None):
        self._env = env
        self._widget_overrides = widget_overrides or {}
        self._config = self._load_yaml(yaml_path, env)
        self._cache = {}

    def _load_yaml(self, yaml_path, env):
        try:
            with open(yaml_path, 'r') as f:
                all_config = yaml.safe_load(f)
            config = all_config.get('dev', {}).copy()
            if env != 'dev' and env in all_config:
                config.update(all_config[env])
            return {k.upper(): v for k, v in config.items()}
        except FileNotFoundError:
            print(f"⚠️  Config file not found: {yaml_path}")
            return {}

    def __getattr__(self, name):
        name_upper = name.upper()
        cache = object.__getattribute__(self, '_cache')
        if name_upper in cache:
            return cache[name_upper]
        overrides = object.__getattribute__(self, '_widget_overrides')
        config = object.__getattribute__(self, '_config')
        if name_upper in overrides:
            value = overrides[name_upper]
        elif name_upper in config:
            value = config[name_upper]
        else:
            raise AttributeError(f"Config parameter '{name}' not found in YAML or widgets")
        if name_upper == 'DEMO_WEEK_START' and value in (None, 'auto', ''):
            value = (datetime.utcnow() - timedelta(days=6)).strftime('%Y-%m-%d')
            print(f"[CONFIG] demo_week_start auto-resolved to {value} (today_utc - 6 days)")
        cache[name_upper] = value
        return value

_dws_override = dbutils.widgets.get("DEMO_WEEK_START").strip()
cfg = Config(
    dbutils.widgets.get("CONFIG_FILE"),
    "dev",
    {"DEMO_WEEK_START": _dws_override} if _dws_override else {},
)
print(f"[CONFIG] cfg.demo_week_start = {cfg.demo_week_start}")

In [ ]:
%run ./_device_model_spec

In [ ]:
from pyspark.sql.functions import (
    when, col, lit, lag, lead, min as spark_min, rand, coalesce, expr
)
from pyspark.sql.window import Window

def make_device_firmware():
    # Base incident table
    df = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.pseudo_incident_7d_labeled")

    # Patient-device mapping
    patient_device_df = (
        spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.patient_device")
        .select("patient_id", "device_id")
        .dropDuplicates(["patient_id", "device_id"])
    )

    # Join to get patient_device.device_id by patient_id
    df = df.join(patient_device_df, on="patient_id", how="left")

    # device_model: deterministic function of patient_id via the shared
    # _device_model_spec (%run above) — IDENTICAL to the registry + 05. Previously this
    # was an independent per-device_id rand(seed=42) draw, which disagreed with the
    # registry's device_model for ~82% of rows (why transformations.sql takes
    # device_model from the registry, not telemetry). Now they agree by construction.
    df = df.withColumn("device_model", expr(device_model_case_sql("patient_id")))

    # ---------------------------------------------------------------------
    # FLEET-WIDE firmware rollout (decoupled from incidents).
    # A firmware version is a property of the rollout TIMELINE, not of whether a
    # device happened to drift: every device goes 3.14 -> 4.0 -> 4.1.
    #   3.14 : baseline (before rollout)
    #   4.0  : the buggy rollout. Window spans BOTH incident windows so each
    #          cohort's calibration fault occurs while on 4.0 (was previously a
    #          single transient aligned to window 1 only, which left the window-2
    #          under-read cohort sitting on the already-"fixed" version).
    #   4.1  : recalled / fixed (after the second incident window ends).
    # Derived from the SAME cfg incident params 05 uses, so the 4.0 window always
    # covers window 1 (over-read) and window 2 (under-read) as the demo window slides.
    from datetime import datetime, timedelta
    _d0 = datetime.fromisoformat(cfg.demo_week_start)
    _rollout_start = (_d0 + timedelta(days=cfg.incident_start_day,
                                      hours=cfg.incident_start_hour)).isoformat() + "+00:00"
    _recall_start  = (_d0 + timedelta(days=cfg.second_incident_start_day,
                                      hours=cfg.second_incident_start_hour,
                                      minutes=cfg.second_incident_duration_min)).isoformat() + "+00:00"

    # Fleet-wide firmware version by time (independent of has_incident):
    firmware_version = (
        when(col("time") < lit(_rollout_start), lit(3.14))
        .when(col("time") < lit(_recall_start), lit(4.0))
        .otherwise(lit(4.1))
    )

    base_df = df.select(
        col("patient_id"),
        col("device_id"),
        col("device_model"),
        col("time").alias("start_time"),
        firmware_version.alias("firmware_version"),
    )

    # Keep only firmware-CHANGE rows per device (start of each version interval),
    # for EVERY device. Firmware is fleet-wide, so each device traverses
    # 3.14 -> 4.0 -> 4.1 and emits up to 3 interval rows.
    _fw_window = Window.partitionBy("device_id").orderBy("start_time")
    result_df = base_df.withColumn(
        "prev_firmware_version", lag("firmware_version").over(_fw_window)
    ).filter(
        (col("firmware_version") != col("prev_firmware_version"))
        | col("prev_firmware_version").isNull()
    ).drop("prev_firmware_version")

    # Compute end_time as next start_time per device_id
    window_time = Window.partitionBy("device_id").orderBy("start_time")
    result_df = result_df.withColumn(
        "end_time_raw", lead("start_time").over(window_time)
    )

    # Replace null with a far-future timestamp so between-joins work
    far_future = lit("9999-12-31T23:59:59.000+00:00")
    result_df = result_df.withColumn(
        "end_time", coalesce(col("end_time_raw"), far_future)
    ).drop("end_time_raw")

    # Select final columns
    result_df = result_df.select(
        "patient_id",
        "device_id",
        "device_model",
        "start_time",
        "end_time",
        "firmware_version",
    )

    return result_df

In [0]:
from pyspark.sql.functions import count_distinct, collect_list

raw_data_name = "raw_device_telemetry_stream"
path = f"{volume_base}/{raw_data_name}"  

raw_device_telemetry_stream = make_device_firmware()
raw_device_telemetry_stream.write.mode("overwrite").parquet(path)
display(raw_device_telemetry_stream)